# LLWM embedding-MSE on TPU v5e-8
Select **Settings → Accelerator → TPU v5e-8**, enable Internet, then Run All.

15 compressor layers, six predictor layers, 128-dimensional Gaussian actions; next-token embedding MSE with detached targets and target-KL penalty. Two MPC tasks every 1,000 steps. All TPU work runs in child processes; do not import torch_xla in this notebook kernel.

A complete run requests 6,000 updates (786,432,000 global tokens). Kaggle quota/session limits may interrupt it; save/export output checkpoints and tokenizer to preserve progress. TPU memory and throughput still require validation on actual Kaggle hardware.

In [ ]:
import os, sys, subprocess, pathlib, shutil
ROOT = pathlib.Path('/kaggle/working')
assert ROOT.exists(), 'Run this notebook on Kaggle with TPU v5e-8 selected.'
assert list(pathlib.Path('/dev').glob('accel*')), 'No TPU devices: select TPU v5e-8 in notebook settings.'
REPO = ROOT / 'LLWM'
BRANCH = 'feat/kaggle-embedding-mse'
def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)
if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/RodrickKamsiyonna/LLWM-.git', REPO])
else:
    assert (REPO / '.git').exists(), 'Existing LLWM directory is not a Git checkout.'
    run(['git', '-C', REPO, 'rev-parse', 'HEAD'])
print('Source commit:')
run(['git', '-C', REPO, 'log', '-1', '--oneline'])


In [ ]:
# Isolated environment avoids changing Kaggle's preinstalled packages.
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
UV = [sys.executable, '-m', 'uv']
run(UV + ['venv', '--python', '3.12', str(REPO / '.venv')])
run(UV + ['sync', '--python', '3.12', '--extra', 'tpu', '--frozen'], cwd=REPO)
PYTHON = REPO / '.venv/bin/python'
ENV = dict(os.environ, NANOCHAT_BASE_DIR=str(ROOT / 'llwm_artifacts'),
           NANOCHAT_DTYPE='bfloat16', OMP_NUM_THREADS='1', PYTHONUNBUFFERED='1')
ENV.pop('PJRT_DEVICE', None)
run([PYTHON, '-c', 'import torch, torch_xla; print("torch",torch.__version__,"XLA",torch_xla.__version__)'], env=ENV)


In [ ]:
# Mandatory tiny embedding-MSE optimizer, replica, and MPC check on all eight TPUs.
SMOKE_ID = 'kaggle-embedding-smoke-' + __import__('uuid').uuid4().hex[:8]
run([PYTHON, '-u', '-m', 'scripts.xla_launch', '--expected-world-size=8',
     '--expected-local-devices=8', '--run-id=' + SMOKE_ID, '--launch-id=' + SMOKE_ID,
     '--control-dir=/kaggle/working/checkpoints/control',
     'scripts.llwm_token_smoke', '--embedding-mse'], cwd=REPO, env=ENV)
print('Eight-rank TPU verification passed.')


In [ ]:
# CPU data/tokenizer setup; downloads 170 training shards plus a validation shard.
BASE = pathlib.Path(ENV['NANOCHAT_BASE_DIR'])
if not (BASE / 'tokenizer/tokenizer.pkl').exists():
    run([PYTHON, '-m', 'nanochat.dataset', '-n', '8', '-w', '8'], cwd=REPO, env=ENV)
    run([PYTHON, '-u', '-m', 'scripts.tok_train', '--max-chars=2000000000', '--vocab-size=32768'], cwd=REPO, env=ENV)
run([PYTHON, '-m', 'nanochat.dataset', '-n', '170', '-w', '8'], cwd=REPO, env=ENV)
MANIFEST = BASE / 'dataset-manifest.json'
run([PYTHON, '-u', '-m', 'scripts.dataset_preflight', '--world-size=8', '--workers=4',
     '--output', MANIFEST, '--max-document-chars=16777216', '--max-document-tokens=4194304',
     '--max-tokenizer-chars=16777216', '--tokenizer-threads=4', '--tokenizer-batch-size=128',
     '--log-dir', ROOT / 'preflight-logs'], cwd=REPO, env=ENV)


In [ ]:
# Foreground launch: rank output is also streamed from its files below.
import threading, time
RUN_ID = 'llwm-embedding-mse-kaggle'
ATTEMPT_ID = RUN_ID + '-' + __import__('uuid').uuid4().hex[:8]
TRAIN_ENV = dict(ENV, RUN_ID=RUN_ID, ATTEMPT_ID=ATTEMPT_ID, DATASET_MANIFEST=str(MANIFEST),
                 NUM_ITERATIONS='6000', PLANNING_EVERY='1000', SAVE_EVERY='500')
# To resume: restore the saved checkpoint directory + tokenizer, keep RUN_ID, and set this to 1.
TRAIN_ENV['RESUME_LATEST'] = '0'
LOG = ROOT / (ATTEMPT_ID + '.log')
stop_reader = threading.Event()
def follow_rank_logs():
    offsets = {}
    while not stop_reader.wait(2):
        for path in (REPO / 'logs' / RUN_ID / ATTEMPT_ID).glob('*/*bootstrap.stdout.log'):
            with path.open(errors='replace') as f:
                f.seek(offsets.get(str(path), 0)); chunk = f.read(); offsets[str(path)] = f.tell()
            if chunk: print(chunk, end='', flush=True)
reader = threading.Thread(target=follow_rank_logs, daemon=True); reader.start()
try:
    with LOG.open('w') as log:
        process = subprocess.Popen(['bash', 'runs/kaggle_embedding_mse.sh'], cwd=REPO,
                                   env=TRAIN_ENV, stdout=log, stderr=subprocess.STDOUT)
        result = process.wait()
    if result: print(LOG.read_text()[-12000:]); raise RuntimeError(f'Training exited with status {result}; see {LOG}')
finally:
    time.sleep(3); stop_reader.set(); reader.join(timeout=5)
print('Training finished. Save this version with outputs to retain checkpoints and tokenizer.')
